# Project Observing Time

This notebook is designed to calculate the total time requested for the various targets within the project, summarized in the different time bins required for the proposal.

In [20]:
import numpy as np

In [3]:
class TargetSet:
    def __init__(self):
        self.survey = None
        self.survey_years = []
        self.survey_semesters = {2023: ['B'], 
                                 2024: ['A', 'B'],
                                 2025: ['A', 'B'],
                                 2026: ['A']}
        self.semester_fraction = []
        self.n_planet_events_per_year = 0
        self.n_stellar_events_per_year = 0
        self.n_bh_events_per_year = 0
        
        self.sinistro_time_per_planet = 0
        self.sinistro_time_per_star = 0
        self.sinistro_time_per_bh = 0
        
        self.floyds_frac = 0.0
        self.floyds_time_per_target = 3.83 # hrs
        
        self.time_per_caustic = 6.0*2.0 # hrs, with two caustic crossings per event
        self.time_per_anomaly = 4.0 # hrs
        self.vis_fraction = 0.0
    
    def calc_total_n_events(self):
        """Calculate the total number of events over the lifetime of the survey operations"""
        self.n_planet_events = 0
        self.n_stellar_events = 0
        self.n_bh_events = 0 
        
        for i,year in enumerate(self.survey_semesters.keys()):
            if len(self.survey_semesters[year]) == 2:
                self.n_planet_events += self.n_planet_events_per_year
                self.n_stellar_events += self.n_stellar_events_per_year
                self.n_bh_events += self.n_bh_events_per_year
                
            elif len(self.survey_semesters[year]) == 1:
                if self.survey_semesters[year][0] == 'A':
                    self.n_planet_events += self.n_planet_events_per_year * self.semester_fraction[1]
                    self.n_stellar_events += self.n_stellar_events_per_year * self.semester_fraction[1]
                    self.n_bh_events += self.n_bh_events_per_year * self.semester_fraction[1]
                else:
                    self.n_planet_events += self.n_planet_events_per_year * self.semester_fraction[0]
                    self.n_stellar_events += self.n_stellar_events_per_year * self.semester_fraction[0]
                    self.n_bh_events += self.n_bh_events_per_year * self.semester_fraction[0]
                
        print('Expect ~'+str(self.n_planet_events_per_year)+' planetary events per year, and '+str(self.n_planet_events)+' over the remaining survey lifetime')
        print('Expect ~'+str(self.n_stellar_events_per_year)+' stellar events per year, and '+str(self.n_stellar_events)+' over the remaining survey lifetime')
        print('Expect ~'+str(self.n_bh_events_per_year)+' stellar remnant events per year, and '+str(self.n_bh_events)+' over the remaining survey lifetime')
      
    def calc_sinistro_time(self):
        """Calculate the time required on the 1m/Sinistro network for different categories of events"""
        
        self.sinistro_time_planet_per_year = self.sinistro_time_per_planet * self.n_planet_events_per_year
        self.sinistro_time_stellar_per_year = self.sinistro_time_per_star * self.n_stellar_events_per_year
        self.sinistro_time_bh_per_year = self.sinistro_time_per_bh * self.n_bh_events_per_year
        print('Time on 1m/Sinistro network required for planetary events per year = '+str(self.sinistro_time_planet_per_year)+'hrs')
        print('Time on 1m/Sinistro network required for stellar events per year = '+str(self.sinistro_time_stellar_per_year)+'hrs')
        print('Time on 1m/Sinistro network required for stellar remnant events per year = '+str(self.sinistro_time_bh_per_year)+'hrs')
        
        self.sinistro_time_planet = []
        self.sinistro_time_stellar = []
        self.sinistro_time_bh = []
        self.sinistro_time = []
        for i,year in enumerate(self.survey_years):
            #self.sinistro_time_planet.append(self.sinistro_time_planet_per_year)
            #self.sinistro_time_stellar.append(self.sinistro_time_stellar_per_year)
            #self.sinistro_time_bh.append(self.sinistro_time_bh_per_year)
            
            time_year = self.sinistro_time_planet_per_year \
                        + self.sinistro_time_stellar_per_year \
                        + self.sinistro_time_bh_per_year
            print('Total time on 1m/Sinistro in year '+str(year)+': '+str(time_year))
            self.sinistro_time.append(time_year)
        
        (time_planet,time_stellar,time_bh,time_total) = self.calc_instrument_time_per_semester('1m/Sinistro',
                                                                                    self.sinistro_time_planet_per_year, 
                                                                                    self.sinistro_time_stellar_per_year, 
                                                                                    self.sinistro_time_bh_per_year)
        self.sinistro_time_planet = time_planet
        self.sinistro_time_stellar = time_stellar
        self.sinistro_time_bh = time_bh
        self.sinistro_time = time_total
        
    def calc_instrument_time_per_semester(self,inst_name,inst_time_planet_per_year, 
                                          inst_time_stellar_per_year, inst_time_bh_per_year,
                                          planets=True, stellar=True):
        
        inst_time_planet = []
        inst_time_stellar = []
        inst_time_bh = []
        inst_time_total = []
        for i,year in enumerate(self.survey_semesters.keys()):
            for s in self.survey_semesters[year]:
                if s == 'A':
                    if planets:
                        inst_time_planet.append(inst_time_planet_per_year*self.semester_fraction[1])
                    else:
                        inst_time_planet.append(0.0)
                    if stellar:
                        inst_time_stellar.append(inst_time_stellar_per_year*self.semester_fraction[1])
                    else:
                        inst_time_stellar.append(0.0)
                    inst_time_bh.append(inst_time_bh_per_year*self.semester_fraction[1])
                else:
                    if planets:
                        inst_time_planet.append(inst_time_planet_per_year*self.semester_fraction[0])
                    else:
                        inst_time_planet.append(0.0)
                    if stellar:
                        inst_time_stellar.append(inst_time_stellar_per_year*self.semester_fraction[0])
                    else:
                        inst_time_stellar.append(0.0)
                    inst_time_bh.append(inst_time_bh_per_year*self.semester_fraction[0])
                
                time_semester = inst_time_planet[-1]+inst_time_stellar[-1]+inst_time_bh[-1]
                inst_time_total.append(time_semester)
                print('Time on '+inst_name+' in semester '+str(year)+s+': '+str(time_semester))

        return inst_time_planet, inst_time_stellar, inst_time_bh, inst_time_total
                
    def calc_floyds_time(self, planets=True, stellar=True):
        """Calculate the time required for 2m/FLOYDS spectroscopy"""
        
        self.floyds_bh_events_per_year = int(round(self.floyds_frac*self.n_bh_events_per_year,0))
        print('Number of stellar remnant events bright enough for FLOYDS per year = '+str(self.floyds_bh_events_per_year))
        self.floyds_time_bh_per_year = int(round((self.floyds_bh_events_per_year * self.floyds_time_per_target),0))
        print('Time on 2m/FLOYDS network required for stellar remnant events per year = '+str(self.floyds_time_bh_per_year)+'hrs')

        if planets:
            self.floyds_planet_events_per_year = int(round(self.floyds_frac*self.n_planet_events_per_year,0))
            print('Number of planetary events bright enough for FLOYDS per year = '+str(self.floyds_planet_events_per_year))
            
            self.floyds_time_planet_per_year = int(round((self.floyds_planet_events_per_year * self.floyds_time_per_target),0))
            print('Time on 2m/FLOYDS network required for planetary events per year = '+str(self.floyds_time_planet_per_year)+'hrs')
        else:
            self.floyds_time_planet_per_year = 0
            
        if stellar:
            self.floyds_stellar_events_per_year = int(round(self.floyds_frac*self.n_stellar_events_per_year,0))
            print('Number of stellar events bright enough for FLOYDS per year = '+str(self.floyds_stellar_events_per_year))
        
            self.floyds_time_stellar_per_year = int(round((self.floyds_stellar_events_per_year * self.floyds_time_per_target),0))
            print('Time on 2m/FLOYDS network required for stellar events per year = '+str(self.floyds_time_stellar_per_year)+'hrs')
        else:
            self.floyds_time_stellar_per_year = 0
            
        self.floyds_time_planet = []
        self.floyds_time_stellar = []
        self.floyds_time_bh = []
        self.floyds_time = []
        for i,year in enumerate(self.survey_years):
            time_year = self.floyds_time_bh_per_year
            if planets:
                time_year += self.floyds_time_planet_per_year
            if stellar:
                time_year += self.floyds_time_stellar_per_year
                
            self.floyds_time.append(time_year)
            print('Total time on 2m/FLOYDS in year '+str(year)+': '+str(time_year))

        (time_planet,time_stellar,time_bh,time_total) = self.calc_instrument_time_per_semester('2m/FLOYDS',
                                                                                    self.floyds_time_planet_per_year, 
                                                                                    self.floyds_time_stellar_per_year, 
                                                                                    self.floyds_time_bh_per_year,
                                                                                    planets=planets,
                                                                                    stellar=stellar)
        self.floyds_time_planet = time_planet
        self.floyds_time_stellar = time_stellar
        self.floyds_time_bh = time_bh
        self.floyds_time = time_total

    def calc_2m_imagers_time(self, planets=True, stellar=True):
        """Estimate of time required in hours on the 2m/MusCAT or 2m/Spectral network to 
        observe caustic crossings for stellar & planetary binary events in RAPID RESPONSE MODE"""

        self.time_rr_stellar_events = []
        self.time_rr_planet_events = []
        self.time_2m_imagers = []
        for i,year in enumerate(self.survey_years):
            time_year = 0.0
            if stellar:
                n_stellar_per_year = int(round((self.n_stellar_events_per_year * self.vis_fraction),0))
                time_stellar_year = n_stellar_per_year * self.time_per_caustic
                self.time_rr_stellar_events.append(time_stellar_year)
                time_year += time_stellar_year
                print('Time on 2m/imagers for '+str(n_stellar_per_year)+' stellar events in year '+str(year)+': '+str(time_stellar_year))
            else:
                self.time_rr_stellar_events.append(0.0)
            
            if planets:
                n_planet_per_year = int(round((self.n_planet_events_per_year * self.vis_fraction),0))
                time_planet_year = n_planet_per_year * self.time_per_caustic
                self.time_rr_planet_events.append(time_planet_year)
                time_year += time_planet_year
                print('Time on 2m/imagers for '+str(n_planet_per_year)+' planetary events in year '+str(year)+': '+str(time_planet_year))
            else:
                self.time_rr_planet_events.append(0.0)
                
            self.time_2m_imagers.append(time_year)
            print('Total time on 2m/imagers in RR mode in year '+str(year)+': '+str(time_year))
    
        n_planet_per_year = int(round((self.n_planet_events_per_year * self.vis_fraction),0))
        time_planet_year = n_planet_per_year * self.time_per_caustic
        n_stellar_per_year = int(round((self.n_stellar_events_per_year * self.vis_fraction),0))
        time_stellar_year = n_stellar_per_year * self.time_per_caustic
        (time_planet,time_stellar,time_bh,time_total) = self.calc_instrument_time_per_semester('2m/imagers',
                                                                                    time_planet_year, 
                                                                                    time_stellar_year, 
                                                                                    0,
                                                                                    planets=planets,
                                                                                    stellar=stellar)
        self.time_rr_planet_events = time_planet
        self.time_rr_stellar_events = time_stellar
        self.time_2m_imagers = time_total
    
    def calc_1m_rr_time(self, planets=True, stellar=True):
        """Time required in Rapid Reponse mode from the 1m network is equivalent to the fraction of the time 
        dedicated to observing anomalies and caustic crossings that isn't already covered by the 2m. 
        
        It is only calculated for the planetary and stellar binary events to be followed from each survey. 
        
        """
        
        # The fraction of the Rapid Response time from the 1m network
        rr_fraction = 1.0 - self.vis_fraction
        
        self.time_rr_stellar_events_1m = []
        self.time_rr_planet_events_1m = []
        self.time_rr_1m_imagers = []
        for i,year in enumerate(self.survey_years):
            time_year = 0.0
            if stellar:
                n_stellar_per_year = int(round((self.n_stellar_events_per_year * rr_fraction),0))
                time_stellar_year = n_stellar_per_year * self.time_per_caustic
                self.time_rr_stellar_events_1m.append(time_stellar_year)
                time_year += time_stellar_year
                print('Rapid response time on 1m/imagers for '+str(n_stellar_per_year)+' stellar events in year '+str(year)+': '+str(time_stellar_year))
            else:
                self.time_rr_stellar_events_1m.append(0.0)
            
            if planets:
                n_planet_per_year = int(round((self.n_planet_events_per_year * rr_fraction),0))
                time_planet_year = n_planet_per_year * self.time_per_caustic
                self.time_rr_planet_events_1m.append(time_planet_year)
                time_year += time_planet_year
                print('Rapid response time on 1m/imagers for '+str(n_planet_per_year)+' planetary events in year '+str(year)+': '+str(time_planet_year))
            else:
                self.time_rr_planet_events_1m.append(0.0)
                
            self.time_rr_1m_imagers.append(time_year)
            print('Total time on 1m/imagers in RR mode in year '+str(year)+': '+str(time_year))
        
        
        if stellar:
            n_stellar_per_year = int(round((self.n_stellar_events_per_year * rr_fraction),0))
            time_stellar_year = n_stellar_per_year * self.time_per_caustic
        else:
            time_stellar_year = 0.0
        
        if planets:
            n_planet_per_year = int(round((self.n_planet_events_per_year * rr_fraction),0))
            time_planet_year = n_planet_per_year * self.time_per_caustic
        else:
            time_planet_year = 0.0
            
        (time_planet,time_stellar,time_bh,time_total) = self.calc_instrument_time_per_semester('1m/imagers RR',
                                                                                    time_planet_year, 
                                                                                    time_stellar_year, 
                                                                                    0,
                                                                                    planets=planets,
                                                                                    stellar=stellar)
        self.time_rr_stellar_events_1m = time_planet
        self.time_rr_planet_events_1m = time_stellar
        self.time_rr_1m_imagers = time_total
        
    def calc_time_totals(self):
        # Calculate time required over all survey semesters
        self.sinistro_time_total = np.array(self.sinistro_time).sum()
        self.floyds_time_total = np.array(self.floyds_time).sum()
        self.time_2m_imagers_total = np.array(self.time_2m_imagers).sum()
        
        self.sinistro_1m_queue = np.zeros(len(self.sinistro_time))
        s = -1
        for i,year in enumerate(self.survey_semesters.keys()):
            annual_semesters = self.survey_semesters[year]
            
            for sem in annual_semesters:
                s += 1
                self.sinistro_1m_queue[s] = self.sinistro_time[s] - self.time_rr_1m_imagers[s]
                print('Semester '+str(year)+sem+' Sinistro total '+str(self.sinistro_time[i]))
                print(' -> Sinistro queue mode '+str(self.sinistro_1m_queue[s]))
                print(' -> Sinistro RR mode '+str(self.time_rr_1m_imagers[s]))
        
        # Calculate time required per semester in each survey year
        # LCO Semesters run A: Feb 1 - July 31 and B: Aug 1 - Jan 31
        # semester_fraction refers to the ratio of the annual target count that 
        # expected to be found in the [B, A] semesters. 
        # The Key Project proposals run from 2023B - 2026A inclusive
        
        print('\n')
        print('Sinistro time total over project: '+str(self.sinistro_time_total)+'hrs')
        print('FLOYDS time total over project: '+str(self.floyds_time_total)+'hrs')
        print('2m imagers time total over project: '+str(self.time_2m_imagers_total)+'hrs')
        
    def calc_time_per_semester(self, annual_time):
        
        key_project_semesters = {}
        
        for i,year in enumerate(self.survey_semesters.keys()):
            annual_semesters = self.survey_semesters[year]
            
            Atime = annual_time[i] * self.semester_fraction[1]
            Btime = annual_time[i] * self.semester_fraction[0]
            
            semester_time = {}
            for s in annual_semesters:
                if s == 'A':
                    semester_time[s] = Atime
                else:
                    semester_time[s] = Btime
                    
            key_project_semesters[year] = semester_time
            print('Year '+str(year)+': '+repr(key_project_semesters[year]))
        
        return key_project_semesters
        

## Microlensing Observations

This section draws on estimates of the expected number of targets calculated in the notebook estimated_number_of_events and the time required per target is calculated in notebook event_simulator. 

In [12]:
# Number of different categories of events per year
nhigh_mag_events = 30
nstellar_events = 98
nlongte_events = 10

In [13]:
mulens_targets = TargetSet()

In [14]:
mulens_targets.survey_years = [2026, 2027, 2028]
mulens_targets.survey_semesters = {2026: ['B'], 
                                 2027: ['A', 'B'],
                                 2028: ['A']}
mulens_targets.semester_fraction = [1.0, 1.0] # Downscaling of events for visibility already included
mulens_targets.n_planet_events_per_year = nhigh_mag_events
mulens_targets.n_stellar_events_per_year = nstellar_events - mulens_targets.n_planet_events_per_year
mulens_targets.n_bh_events_per_year = nlongte_events
mulens_targets.calc_total_n_events()

Expect ~30 planetary events per year, and 90.0 over the remaining survey lifetime
Expect ~68 stellar events per year, and 204.0 over the remaining survey lifetime
Expect ~10 stellar remnant events per year, and 30.0 over the remaining survey lifetime


In [15]:
# Estimates of time required in hours on the 1m/Sinistro network to observe the different categories of events
mulens_targets.sinistro_time_per_planet = 56.0
mulens_targets.sinistro_time_per_star = 33.0
mulens_targets.sinistro_time_per_bh = 13.0
mulens_targets.calc_sinistro_time()

Time on 1m/Sinistro network required for planetary events per year = 1680.0hrs
Time on 1m/Sinistro network required for stellar events per year = 2244.0hrs
Time on 1m/Sinistro network required for stellar remnant events per year = 130.0hrs
Total time on 1m/Sinistro in year 2026: 4054.0
Total time on 1m/Sinistro in year 2027: 4054.0
Total time on 1m/Sinistro in year 2028: 4054.0
Time on 1m/Sinistro in semester 2026B: 4054.0
Time on 1m/Sinistro in semester 2027A: 4054.0
Time on 1m/Sinistro in semester 2027B: 4054.0
Time on 1m/Sinistro in semester 2028A: 4054.0


In [16]:
# Estimates of time required in hours on the 2m/FLOYDS network to observe the different categories of events
mulens_targets.floyds_frac = 0.994 # Fraction of events likely to be bright enough for FLOYDS
mulens_targets.calc_floyds_time(stellar=False)

Number of stellar remnant events bright enough for FLOYDS per year = 10
Time on 2m/FLOYDS network required for stellar remnant events per year = 38hrs
Number of planetary events bright enough for FLOYDS per year = 30
Time on 2m/FLOYDS network required for planetary events per year = 115hrs
Total time on 2m/FLOYDS in year 2026: 153
Total time on 2m/FLOYDS in year 2027: 153
Total time on 2m/FLOYDS in year 2028: 153
Time on 2m/FLOYDS in semester 2026B: 153.0
Time on 2m/FLOYDS in semester 2027A: 153.0
Time on 2m/FLOYDS in semester 2027B: 153.0
Time on 2m/FLOYDS in semester 2028A: 153.0


In [17]:
# Estimate of time required in hours on the 2m/MusCAT or 2m/Spectral network to 
# observe high mag events in RAPID RESPONSE MODE
# Note 'stellar' for this function refers to all single stellar events
mulens_targets.vis_fraction = 8.0 / 24.0
mulens_targets.calc_2m_imagers_time(stellar=False)

Time on 2m/imagers for 10 planetary events in year 2026: 120.0
Total time on 2m/imagers in RR mode in year 2026: 120.0
Time on 2m/imagers for 10 planetary events in year 2027: 120.0
Total time on 2m/imagers in RR mode in year 2027: 120.0
Time on 2m/imagers for 10 planetary events in year 2028: 120.0
Total time on 2m/imagers in RR mode in year 2028: 120.0
Time on 2m/imagers in semester 2026B: 120.0
Time on 2m/imagers in semester 2027A: 120.0
Time on 2m/imagers in semester 2027B: 120.0
Time on 2m/imagers in semester 2028A: 120.0


In [18]:
# Estimate the remaining Rapid Response time required from the 1m network
mulens_targets.calc_1m_rr_time(stellar=False)

Rapid response time on 1m/imagers for 20 planetary events in year 2026: 240.0
Total time on 1m/imagers in RR mode in year 2026: 240.0
Rapid response time on 1m/imagers for 20 planetary events in year 2027: 240.0
Total time on 1m/imagers in RR mode in year 2027: 240.0
Rapid response time on 1m/imagers for 20 planetary events in year 2028: 240.0
Total time on 1m/imagers in RR mode in year 2028: 240.0
Time on 1m/imagers RR in semester 2026B: 240.0
Time on 1m/imagers RR in semester 2027A: 240.0
Time on 1m/imagers RR in semester 2027B: 240.0
Time on 1m/imagers RR in semester 2028A: 240.0


In [21]:
mulens_targets.calc_time_totals()

Semester 2026B Sinistro total 4054.0
 -> Sinistro queue mode 3814.0
 -> Sinistro RR mode 240.0
Semester 2027A Sinistro total 4054.0
 -> Sinistro queue mode 3814.0
 -> Sinistro RR mode 240.0
Semester 2027B Sinistro total 4054.0
 -> Sinistro queue mode 3814.0
 -> Sinistro RR mode 240.0
Semester 2028A Sinistro total 4054.0
 -> Sinistro queue mode 3814.0
 -> Sinistro RR mode 240.0


Sinistro time total over project: 16216.0hrs
FLOYDS time total over project: 612.0hrs
2m imagers time total over project: 480.0hrs


## Cataclysmic Variables


In [22]:
ncvs = 5
obs_time = 3.0 # hrs per target continuous imaging
sinistro_time = ncvs * obs_time

print('Sinistro time: ' + str(sinistro_time) + 'hrs')

# This needs to be broken down per semester

Sinistro time: 15.0hrs


## Black Hole X-ray Binaries